In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
  "font.family": "serif",
  "font.serif": ["Times New Roman", "DejaVu Serif"],
  "mathtext.fontset": "stix",
  "font.size": 18,
  "axes.labelsize": 18,
  "legend.fontsize": 18,
  "xtick.labelsize": 18,
  "ytick.labelsize": 18,
  "figure.dpi": 150,
  "savefig.dpi": 300,
})

In [ ]:
def build_filleted_path(
  waypoints: np.ndarray,
  fillet_radius: float,
  sample_spacing: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
  """Builds a planar path of straight legs joined by circular fillets of one radius.

  The polyline through `waypoints` is rounded at every interior corner by a circular
  arc of radius `r` tangent to both adjacent legs. The curve is continuous in position
  and tangent, and its curvature is piecewise constant, equal to zero on the straight
  parts and to `±1/r` on the arcs, with the sign given by the turning direction. The
  largest curvature of the path is therefore exactly `1/r`, so the tightest bend is
  prescribed directly rather than emerging from an interpolant.

  Algorithm.
    1. Compute the leg vectors `ℓ_j = P_{j+1} − P_j`, their lengths `|ℓ_j|`, and
       their unit directions `d_j = ℓ_j / |ℓ_j|`.
    2. At the corner `P_{j+1}` between legs `j` and `j+1`, compute the signed turning
       angle `φ_j = atan2(d_j × d_{j+1}, d_j · d_{j+1})`, which is positive for a
       counterclockwise turn.
    3. Compute the tangent length `τ_j = r tan(|φ_j| / 2)`, the distance from the
       corner to each point where the arc meets a leg.
    4. Trim leg `j` by `τ_{j−1}` at its start and by `τ_j` at its end, with zero trim
       at the two ends of the path, and require `τ_{j−1} + τ_j < |ℓ_j|` so that every
       leg keeps a straight part of positive length.
    5. Sample the straight part of leg `j` from `P_j + τ_{j−1} d_j` to the arc start
       `a_j = P_{j+1} − τ_j d_j`.
    6. Place the arc center at `c_j = a_j + sign(φ_j) r n_j`, where
       `n_j = (−d_{j,y}, d_{j,x})` is the left normal of `d_j`, and sample the arc as
       `c_j + R(θ)(a_j − c_j)` for `θ ∈ (0, φ_j]`, where `R(θ)` is the
       counterclockwise rotation by `θ`.
    7. Concatenate all samples without repeating shared end points and compute the
       arc coordinate as the cumulative chord length. On the arcs this converges to
       the exact arc length with relative error `O(Δ² / r²)` for sample spacing `Δ`.

  Args:
    waypoints: Array of shape `(num_waypoints, 2)` of polyline corner points ordered
      from the clamped end to the free end. Consecutive points must be distinct and
      no corner may reverse direction exactly, so `|φ_j| < π`.
    fillet_radius: Radius `r` of every fillet in the length unit of `waypoints`.
      Positive and small enough to satisfy the condition of step 4.
    sample_spacing: Target distance `Δ` between consecutive samples. Positive and
      typically `r / 40` or smaller.

  Returns:
    A tuple `(arc, points, curvatures)`. `arc` has shape `(num_samples,)` and holds
      the arc coordinate `s`, starting at zero on the first waypoint. `points` has
      shape `(num_samples, 2)` and holds the sampled positions. `curvatures` has
      shape `(num_samples,)` and holds the signed curvature `κ(s)` of the piece each
      sample belongs to, in inverse length units.
  """
  legs = np.diff(waypoints, axis=0)
  leg_lengths = np.linalg.norm(legs, axis=-1)
  leg_dirs = legs / leg_lengths[:, None]
  sin_turn = leg_dirs[:-1, 0] * leg_dirs[1:, 1] - leg_dirs[:-1, 1] * leg_dirs[1:, 0]
  cos_turn = np.einsum("jd,jd->j", leg_dirs[:-1], leg_dirs[1:])
  turn_angles = np.arctan2(sin_turn, cos_turn)
  tangent_lengths = fillet_radius * np.tan(np.abs(turn_angles) / 2.0)
  trim_start = np.concatenate([[0.0], tangent_lengths])
  trim_end = np.concatenate([tangent_lengths, [0.0]])
  assert np.all(trim_start + trim_end < leg_lengths), "Fillet radius too large."

  points, curvatures = [waypoints[:1]], [np.zeros(1)]
  for leg in range(leg_lengths.shape[0]):
    line_start = waypoints[leg] + trim_start[leg] * leg_dirs[leg]
    arc_start = waypoints[leg + 1] - trim_end[leg] * leg_dirs[leg]
    line_length = leg_lengths[leg] - trim_start[leg] - trim_end[leg]
    num_line = int(np.ceil(line_length / sample_spacing))
    frac = np.linspace(start=0.0, stop=1.0, num=num_line + 1)[1:, None]
    points.append(line_start + frac * (arc_start - line_start))
    curvatures.append(np.zeros(num_line))
    if leg == leg_lengths.shape[0] - 1:
      break

    turn = turn_angles[leg]
    left_normal = np.array([-leg_dirs[leg, 1], leg_dirs[leg, 0]])
    arc_center = arc_start + np.sign(turn) * fillet_radius * left_normal
    radius_vec = arc_start - arc_center
    num_arc = int(np.ceil(fillet_radius * abs(turn) / sample_spacing))
    sweep = np.linspace(start=0.0, stop=turn, num=num_arc + 1)[1:]
    rotated = np.stack(
      [
        np.cos(sweep) * radius_vec[0] - np.sin(sweep) * radius_vec[1],
        np.sin(sweep) * radius_vec[0] + np.cos(sweep) * radius_vec[1],
      ],
      axis=-1,
    )
    points.append(arc_center + rotated)
    curvatures.append(np.full(shape=num_arc, fill_value=np.sign(turn) / fillet_radius))

  points = np.concatenate(points)
  curvatures = np.concatenate(curvatures)
  chord_lengths = np.linalg.norm(np.diff(points, axis=0), axis=-1)
  arc = np.concatenate([[0.0], np.cumsum(chord_lengths)])
  return arc, points, curvatures

In [ ]:
# The length unit must match the rod builder. The value below is in meters.
letter_height = 20e-3
# Horizontal positions of the letter strokes, in units of the letter height.
n_left = 0.5
n_right = n_left + 1.0
u_left = n_right + 0.4
u_right = u_left + 0.8
tail_end = u_right + 0.25
# The path leaves the clamp heading along +x, the tangent of the straight dry rod, so
# the target and the rod agree at s = 0.
waypoints = letter_height * np.array([
  [0.0, 0.0],
  [n_left, 0.0],
  [n_left, 1.4],
  [n_right, -0.4],
  [n_right, 1.0],
  [u_left, 1.0],
  [u_left, 0.0],
  [u_right, 0.0],
  [u_right, 1.0],
  [tail_end, 1.0],
])
fillet_radius = 0.15 * letter_height

arc, points, curvatures = build_filleted_path(
  waypoints=waypoints, fillet_radius=fillet_radius, sample_spacing=fillet_radius / 20
)
print(f"Target length L = {arc[-1]:.4g}")
print(f"Peak |kappa L| = {np.abs(curvatures).max() * arc[-1]:.3g}")

In [ ]:
target_df = pd.DataFrame(
  {"s": arc, "x": points[:, 0], "y": points[:, 1], "z": np.zeros_like(arc)}
)
target_df.to_csv("crops/data/targets/nu.csv", index=False)

In [ ]:
total_length = arc[-1]
fig, axes = plt.subplots(
  nrows=1, ncols=1, figsize=(6.5, 5.0)
)
# axes.plot(
#   waypoints[:, 0], waypoints[:, 1], ls=":", marker="o", ms=3, lw=0.8,
#   color="0.6", label="Waypoints",
# )
axes.plot(points[:, 0], points[:, 1], color="k", lw=1.5, label="Target")
axes.plot(
  points[0, 0], points[0, 1], marker="s", ms=6, color="C3", ls="none",
  label="Clamp",
)

axes.set_aspect("equal")
# axes.set_xlabel(r"$x$")
# axes.set_ylabel(r"$y$")
# axes.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0))
axes.spines[["left", "right", "top", "bottom"]].set_visible(False)
axes.set_xticks([])
axes.set_yticks([])
fig.tight_layout()
fig.savefig("results/paper/swell_nu/nu_target.pdf")

In [ ]:
fig, axes = plt.subplots(
  nrows=1, ncols=1, figsize=(6.5, 5.0)
)
axes.plot(arc / total_length, curvatures * total_length, color="k", lw=1.2)
axes.axhline(0.0, color="0.7", lw=0.6)
axes.set_xlim(0.0, 1.0)
axes.set_xlabel(r"Normalized arc length $s/L$")
axes.set_ylabel(r"Normalized curvature $\kappa L$")